# Libs

In [1]:
import os
import gc
import timeit
import pathlib
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import accuracy_score, recall_score

# File Paths

In [2]:
MODELS_DIR = pathlib.Path("../models/keras")
RESULTS_DIR = pathlib.Path("../results")
DATA_PATH = pathlib.Path("../data/fdia_dataset_processed.npz")

RESULTS_DIR.mkdir(exist_ok=True)

print("Models directory:", MODELS_DIR.resolve())
print("Results directory:", RESULTS_DIR.resolve())
print("Dataset:", DATA_PATH.resolve())

print("\nDataset exists:", DATA_PATH.exists())
print("Models directory exists:", MODELS_DIR.exists())

Models directory: C:\tinyml-fdia-windows\models\keras
Results directory: C:\tinyml-fdia-windows\results
Dataset: C:\tinyml-fdia-windows\data\fdia_dataset_processed.npz

Dataset exists: True
Models directory exists: True


# Dataset

In [3]:
data = np.load(DATA_PATH)

X_test = data["X_test"].astype(np.float32)
y_test = data["y_test"]

# LSTM input: (samples, timesteps, features)
X_test_lstm = np.transpose(X_test, (0, 2, 1))

print("X_test shape:", X_test.shape)
print("X_test LSTM shape:", X_test_lstm.shape)
print("y_test shape:", y_test.shape)

X_test shape: (9720, 6, 83)
X_test LSTM shape: (9720, 83, 6)
y_test shape: (9720,)


## Dataset subsample

In [4]:
np.random.seed(42)

sample_size = int(0.25 * len(X_test))

timing_idx = np.random.choice(
    len(X_test),
    size=sample_size,
    replace=False
)

X_lstm_timing = X_test_lstm[timing_idx]

print("Timing fraction: 10%")
print("Timing samples:", len(X_lstm_timing))
print("Timing shape:", X_lstm_timing.shape)

Timing fraction: 10%
Timing samples: 2430
Timing shape: (2430, 83, 6)


# Functions

In [15]:
def measure_inference_time(model, X_timing, runs=100):
    run_times = []

    for _ in range(runs):
        start = timeit.default_timer()

        model.predict(X_timing, batch_size=len(X_timing), verbose=0)

        end = timeit.default_timer()

        total_time_ms = (end - start) * 1000
        mean_time_ms = total_time_ms / len(X_timing)
        run_times.append(mean_time_ms)

    final_mean_ms = np.mean(run_times)

    print(f"Runs: {runs}")
    print(f"Samples per run: {len(X_timing)}")
    print(f"Average inference time per sample: {final_mean_ms:.6f} ms")

    return final_mean_ms

def save_weights_as_npz(model, file_path):
    all_weights = {}

    for i, layer in enumerate(model.layers):
        weights = layer.get_weights()

        if weights:
            for j, weight in enumerate(weights):
                all_weights[f"layer_{i}_weight_{j}"] = weight

    np.savez(file_path, **all_weights)


def get_architecture_size_kb(model):
    architecture_json = model.to_json()
    architecture_size_bytes = len(architecture_json.encode("utf-8"))

    return architecture_size_bytes / 1024


# def get_model_size_kb(model, weights_path):
#     save_weights_as_npz(model, weights_path)

#     weights_size_kb = os.path.getsize(weights_path) / 1024
#     architecture_size_kb = get_architecture_size_kb(model)

#     return weights_size_kb + architecture_size_kb

def get_model_size_kb(model, weights_path=None):
    total_bytes = 0

    for weight in model.weights:
        num_params = np.prod(weight.shape)
        bytes_per_param = np.dtype(weight.dtype).itemsize
        total_bytes += num_params * bytes_per_param

    return total_bytes / 1024

# Result List

In [10]:
results = []

# LSTM TESTS

## LSTM Original

In [7]:
print("Testing: LSTM_Original")

model_path = MODELS_DIR / "LSTM_HPO.keras"
weights_path = RESULTS_DIR / "LSTM_Original_weights.npz"

model = tf.keras.models.load_model(model_path)

# Full test set evaluation
y_prob = model.predict(X_test_lstm, verbose=0)
y_pred = (y_prob > 0.5).astype(int).flatten()

accuracy = accuracy_score(y_test, y_pred)
fdia_recall = recall_score(y_test, y_pred, pos_label=1)
fault_recall = recall_score(y_test, y_pred, pos_label=0)

# Model information
parameters = model.count_params()
model_size_kb = get_model_size_kb(model, weights_path)

# Inference timing
inference_time_ms = measure_inference_time(
    model,
    X_lstm_timing
)

result = {
    "model": "LSTM_Original",
    "architecture": "LSTM",
    "parameters": parameters,
    "accuracy": accuracy,
    "fdia_recall": fdia_recall,
    "fault_recall": fault_recall,
    "model_size_kb": model_size_kb,
    "inference_time_ms": inference_time_ms
}

results.append(result)

print("\n=== RESULT ===")
print(pd.DataFrame([result]).to_string(index=False))

Testing: LSTM_Original
Runs: 100
Samples per run: 2430
Average inference time per sample: 0.436676 ms

=== RESULT ===
        model architecture  parameters  accuracy  fdia_recall  fault_recall  model_size_kb  inference_time_ms
LSTM_Original         LSTM      159581  0.989198     0.987847      0.990415     639.392578           0.436676


## LSTM NODE

In [8]:
print("Testing: LSTM_Node")

model_path = MODELS_DIR / "LSTM_NodePruned.keras"
weights_path = RESULTS_DIR / "LSTM_Node_weights.npz"

model = tf.keras.models.load_model(model_path)

# Full test set evaluation
y_prob = model.predict(X_test_lstm, verbose=0)
y_pred = (y_prob > 0.5).astype(int).flatten()

accuracy = accuracy_score(y_test, y_pred)
fdia_recall = recall_score(y_test, y_pred, pos_label=1)
fault_recall = recall_score(y_test, y_pred, pos_label=0)

# Model information
parameters = model.count_params()
model_size_kb = get_model_size_kb(model, weights_path)

# Inference timing
inference_time_ms = measure_inference_time(
    model,
    X_lstm_timing
)

result = {
    "model": "LSTM_Node",
    "architecture": "LSTM",
    "parameters": parameters,
    "accuracy": accuracy,
    "fdia_recall": fdia_recall,
    "fault_recall": fault_recall,
    "model_size_kb": model_size_kb,
    "inference_time_ms": inference_time_ms
}

results.append(result)

print("\n=== RESULT ===")
print(pd.DataFrame([result]).to_string(index=False))

Testing: LSTM_Node
Runs: 100
Samples per run: 2430
Average inference time per sample: 0.414895 ms

=== RESULT ===
    model architecture  parameters  accuracy  fdia_recall  fault_recall  model_size_kb  inference_time_ms
LSTM_Node         LSTM      133697  0.983436     0.996311      0.971831     537.669922           0.414895


## LSTM WEIGHT

In [9]:
print("Testing: LSTM_Weight")

model_path = MODELS_DIR / "LSTM_WeightPruned.keras"
weights_path = RESULTS_DIR / "LSTM_Weight_weights.npz"

model = tf.keras.models.load_model(model_path)

# Full test set evaluation
y_prob = model.predict(X_test_lstm, verbose=0)
y_pred = (y_prob > 0.5).astype(int).flatten()

accuracy = accuracy_score(y_test, y_pred)
fdia_recall = recall_score(y_test, y_pred, pos_label=1)
fault_recall = recall_score(y_test, y_pred, pos_label=0)

# Model information
parameters = model.count_params()
model_size_kb = get_model_size_kb(model, weights_path)

# Inference timing
inference_time_ms = measure_inference_time(
    model,
    X_lstm_timing
)

result = {
    "model": "LSTM_Weight",
    "architecture": "LSTM",
    "parameters": parameters,
    "accuracy": accuracy,
    "fdia_recall": fdia_recall,
    "fault_recall": fault_recall,
    "model_size_kb": model_size_kb,
    "inference_time_ms": inference_time_ms
}

results.append(result)

print("\n=== RESULT ===")
print(pd.DataFrame([result]).to_string(index=False))

Testing: LSTM_Weight


c:\Users\nalui\tensorflow_env\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam', because it has 48 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(store)


Runs: 100
Samples per run: 2430
Average inference time per sample: 0.434284 ms

=== RESULT ===
      model architecture  parameters  accuracy  fdia_recall  fault_recall  model_size_kb  inference_time_ms
LSTM_Weight         LSTM      159581  0.994753     0.994792      0.994718     639.375977           0.434284


## LSTM WEIGHT NODE

In [10]:
print("Testing: LSTM_WeightNode")

model_path = MODELS_DIR / "LSTM_WeightNodePruned.keras"
weights_path = RESULTS_DIR / "LSTM_WeightNode_weights.npz"

model = tf.keras.models.load_model(model_path)

# Full test set evaluation
y_prob = model.predict(X_test_lstm, verbose=0)
y_pred = (y_prob > 0.5).astype(int).flatten()

accuracy = accuracy_score(y_test, y_pred)
fdia_recall = recall_score(y_test, y_pred, pos_label=1)
fault_recall = recall_score(y_test, y_pred, pos_label=0)

# Model information
parameters = model.count_params()
model_size_kb = get_model_size_kb(model, weights_path)

# Inference timing
inference_time_ms = measure_inference_time(
    model,
    X_lstm_timing
)

result = {
    "model": "LSTM_WeightNode",
    "architecture": "LSTM",
    "parameters": parameters,
    "accuracy": accuracy,
    "fdia_recall": fdia_recall,
    "fault_recall": fault_recall,
    "model_size_kb": model_size_kb,
    "inference_time_ms": inference_time_ms
}

results.append(result)

print("\n=== RESULT ===")
print(pd.DataFrame([result]).to_string(index=False))

Testing: LSTM_WeightNode
Runs: 100
Samples per run: 2430
Average inference time per sample: 0.421255 ms

=== RESULT ===
          model architecture  parameters  accuracy  fdia_recall  fault_recall  model_size_kb  inference_time_ms
LSTM_WeightNode         LSTM      149537  0.994547     0.995877      0.993349     599.547852           0.421255


## LSTM Node-Weight

In [11]:
print("Testing: LSTM_NodeWeight")

model_path = MODELS_DIR / "LSTM_NodeWeightPruned.keras"
weights_path = RESULTS_DIR / "LSTM_NodeWeight_weights.npz"

model = tf.keras.models.load_model(model_path)

# Full test set evaluation
y_prob = model.predict(X_test_lstm, verbose=0)
y_pred = (y_prob > 0.5).astype(int).flatten()

accuracy = accuracy_score(y_test, y_pred)
fdia_recall = recall_score(y_test, y_pred, pos_label=1)
fault_recall = recall_score(y_test, y_pred, pos_label=0)

# Model information
parameters = model.count_params()
model_size_kb = get_model_size_kb(model, weights_path)

# Inference timing
inference_time_ms = measure_inference_time(
    model,
    X_lstm_timing
)

result = {
    "model": "LSTM_NodeWeight",
    "architecture": "LSTM",
    "parameters": parameters,
    "accuracy": accuracy,
    "fdia_recall": fdia_recall,
    "fault_recall": fault_recall,
    "model_size_kb": model_size_kb,
    "inference_time_ms": inference_time_ms
}

results.append(result)

print("\n=== RESULT ===")
print(pd.DataFrame([result]).to_string(index=False))

Testing: LSTM_NodeWeight


c:\Users\nalui\tensorflow_env\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam', because it has 48 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(store)


Runs: 100
Samples per run: 2430
Average inference time per sample: 0.403727 ms

=== RESULT ===
          model architecture  parameters  accuracy  fdia_recall  fault_recall  model_size_kb  inference_time_ms
LSTM_NodeWeight         LSTM      133697    0.9893     0.993707      0.985329      538.28125           0.403727


# MLP TESTS

## MLP Original

In [ ]:
print("Testing: MLP_Original")

model_path = MODELS_DIR / "MLP_HPO.keras"
weights_path = RESULTS_DIR / "MLP_Original_weights.npz"

model = tf.keras.models.load_model(model_path)

# Full test set evaluation
y_prob = model.predict(X_test, verbose=0)
y_pred = (y_prob > 0.5).astype(int).flatten()

accuracy = accuracy_score(y_test, y_pred)
fdia_recall = recall_score(y_test, y_pred, pos_label=1)
fault_recall = recall_score(y_test, y_pred, pos_label=0)

# Model information
parameters = model.count_params()
model_size_kb = get_model_size_kb(model, weights_path)

# Inference timing
X_mlp_timing = X_test[timing_idx]

inference_time_ms = measure_inference_time(model,X_mlp_timing)

result = {
    "model": "MLP_Original",
    "architecture": "MLP",
    "parameters": parameters,
    "accuracy": accuracy,
    "fdia_recall": fdia_recall,
    "fault_recall": fault_recall,
    "model_size_kb": model_size_kb,
    "inference_time_ms": inference_time_ms
}

results.append(result)

print("\n=== RESULT ===")
print(pd.DataFrame([result]).to_string(index=False))

Testing: MLP_Original
Runs: 100
Samples per run: 2430
Average inference time per sample: 0.034930 ms

=== RESULT ===
       model architecture  parameters  accuracy  fdia_recall  fault_recall  model_size_kb  inference_time_ms
MLP_Original          MLP       44753  0.998354     0.998264      0.998435     185.208008            0.03493


## MLP Node

In [13]:
print("Testing: MLP_Node")

model_path = MODELS_DIR / "MLP_NodePruned.keras"
weights_path = RESULTS_DIR / "MLP_Node_weights.npz"

model = tf.keras.models.load_model(model_path)

# Full test set evaluation
y_prob = model.predict(X_test, verbose=0)
y_pred = (y_prob > 0.5).astype(int).flatten()

accuracy = accuracy_score(y_test, y_pred)
fdia_recall = recall_score(y_test, y_pred, pos_label=1)
fault_recall = recall_score(y_test, y_pred, pos_label=0)

# Model information
parameters = model.count_params()
model_size_kb = get_model_size_kb(model, weights_path)

# Inference timing
inference_time_ms = measure_inference_time(
    model,
    X_mlp_timing
)

result = {
    "model": "MLP_Node",
    "architecture": "MLP",
    "parameters": parameters,
    "accuracy": accuracy,
    "fdia_recall": fdia_recall,
    "fault_recall": fault_recall,
    "model_size_kb": model_size_kb,
    "inference_time_ms": inference_time_ms
}

results.append(result)

print("\n=== RESULT ===")
print(pd.DataFrame([result]).to_string(index=False))

Testing: MLP_Node
Runs: 100
Samples per run: 2430
Average inference time per sample: 0.033342 ms

=== RESULT ===
   model architecture  parameters  accuracy  fdia_recall  fault_recall  model_size_kb  inference_time_ms
MLP_Node          MLP       28920  0.995988     0.994141      0.997653     122.742188           0.033342


## MLP Weight

In [14]:
print("Testing: MLP_Weight")

model_path = MODELS_DIR / "MLP_WeightPruned.keras"
weights_path = RESULTS_DIR / "MLP_Weight_weights.npz"

model = tf.keras.models.load_model(model_path)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Full test set evaluation
y_prob = model.predict(X_test, verbose=0)
y_pred = (y_prob > 0.5).astype(int).flatten()

accuracy = accuracy_score(y_test, y_pred)
fdia_recall = recall_score(y_test, y_pred, pos_label=1)
fault_recall = recall_score(y_test, y_pred, pos_label=0)

# Model information
parameters = model.count_params()
model_size_kb = get_model_size_kb(model, weights_path)

# Inference timing
inference_time_ms = measure_inference_time(
    model,
    X_mlp_timing
)

result = {
    "model": "MLP_Weight",
    "architecture": "MLP",
    "parameters": parameters,
    "accuracy": accuracy,
    "fdia_recall": fdia_recall,
    "fault_recall": fault_recall,
    "model_size_kb": model_size_kb,
    "inference_time_ms": inference_time_ms
}

results.append(result)

print("\n=== RESULT ===")
print(pd.DataFrame([result]).to_string(index=False))

Testing: MLP_Weight


c:\Users\nalui\tensorflow_env\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam', because it has 30 variables whereas the saved optimizer has 1 variables. 
  saveable.load_own_variables(store)


Runs: 100
Samples per run: 2430
Average inference time per sample: 0.033445 ms

=== RESULT ===
     model architecture  parameters  accuracy  fdia_recall  fault_recall  model_size_kb  inference_time_ms
MLP_Weight          MLP       44753  0.998148     0.998264      0.998044     185.194336           0.033445


## MLP Weight-Node

In [15]:
print("Testing: MLP_WeightNode")

model_path = MODELS_DIR / "MLP_WeightNodePruned.keras"
weights_path = RESULTS_DIR / "MLP_WeightNode_weights.npz"

model = tf.keras.models.load_model(model_path)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Full test set evaluation
y_prob = model.predict(X_test, verbose=0)
y_pred = (y_prob > 0.5).astype(int).flatten()

accuracy = accuracy_score(y_test, y_pred)
fdia_recall = recall_score(y_test, y_pred, pos_label=1)
fault_recall = recall_score(y_test, y_pred, pos_label=0)

# Model information
parameters = model.count_params()
model_size_kb = get_model_size_kb(model, weights_path)

# Inference timing
inference_time_ms = measure_inference_time(
    model,
    X_mlp_timing
)

result = {
    "model": "MLP_WeightNode",
    "architecture": "MLP",
    "parameters": parameters,
    "accuracy": accuracy,
    "fdia_recall": fdia_recall,
    "fault_recall": fault_recall,
    "model_size_kb": model_size_kb,
    "inference_time_ms": inference_time_ms
}

results.append(result)

print("\n=== RESULT ===")
print(pd.DataFrame([result]).to_string(index=False))

Testing: MLP_WeightNode
Runs: 100
Samples per run: 2430
Average inference time per sample: 0.034538 ms

=== RESULT ===
         model architecture  parameters  accuracy  fdia_recall  fault_recall  model_size_kb  inference_time_ms
MLP_WeightNode          MLP       28206  0.986214     0.971354      0.999609     120.561523           0.034538


## MLP Node-Weight

In [16]:
print("Testing: MLP_NodeWeight")

model_path = MODELS_DIR / "MLP_NodeWeightPruned.keras"
weights_path = RESULTS_DIR / "MLP_NodeWeight_weights.npz"

model = tf.keras.models.load_model(model_path)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Full test set evaluation
y_prob = model.predict(X_test, verbose=0)
y_pred = (y_prob > 0.5).astype(int).flatten()

accuracy = accuracy_score(y_test, y_pred)
fdia_recall = recall_score(y_test, y_pred, pos_label=1)
fault_recall = recall_score(y_test, y_pred, pos_label=0)

# Model information
parameters = model.count_params()
model_size_kb = get_model_size_kb(model, weights_path)

# Inference timing
inference_time_ms = measure_inference_time(
    model,
    X_mlp_timing
)

result = {
    "model": "MLP_NodeWeight",
    "architecture": "MLP",
    "parameters": parameters,
    "accuracy": accuracy,
    "fdia_recall": fdia_recall,
    "fault_recall": fault_recall,
    "model_size_kb": model_size_kb,
    "inference_time_ms": inference_time_ms
}

results.append(result)

print("\n=== RESULT ===")
print(pd.DataFrame([result]).to_string(index=False))

Testing: MLP_NodeWeight


c:\Users\nalui\tensorflow_env\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam', because it has 30 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(store)


Runs: 100
Samples per run: 2430
Average inference time per sample: 0.033700 ms

=== RESULT ===
         model architecture  parameters  accuracy  fdia_recall  fault_recall  model_size_kb  inference_time_ms
MLP_NodeWeight          MLP       28920  0.997737     0.997613      0.997848     123.344727             0.0337


# XGBoost

In [17]:
import xgboost as xgb

print("Testing XGBoost models")

X_test_flat = X_test.reshape(X_test.shape[0], -1).astype(np.float32)

xgb_models = {
    "XGBoost_Original": "XGBoost_HPO.json",
    "XGBoost_Pruned": "XGBoost_Pruned.json"
}

for name, filename in xgb_models.items():

    print(f"\nTesting: {name}")

    model_path = MODELS_DIR / filename

    model = xgb.XGBClassifier()
    model.load_model(model_path)

    # Full test set evaluation
    y_pred = model.predict(X_test_flat)

    accuracy = accuracy_score(y_test, y_pred)
    fdia_recall = recall_score(y_test, y_pred, pos_label=1)
    fault_recall = recall_score(y_test, y_pred, pos_label=0)

    # Model size
    model_size_kb = os.path.getsize(model_path) / 1024

    # Inference timing over the full test set
    times = []

    X_xgb_timing = X_test_flat[timing_idx]

    start = timeit.default_timer()
    
    model.predict(X_xgb_timing)
    
    end = timeit.default_timer()
    
    total_time_ms = (end - start) * 1000
    inference_time_ms = total_time_ms / len(X_xgb_timing)

    result = {
        "model": name,
        "architecture": "XGBoost",
        "parameters": np.nan,
        "accuracy": accuracy,
        "fdia_recall": fdia_recall,
        "fault_recall": fault_recall,
        "model_size_kb": model_size_kb,
        "inference_time_ms": inference_time_ms
    }

    results.append(result)

    print("\n=== RESULT ===")
    print(pd.DataFrame([result]).to_string(index=False))

Testing XGBoost models

Testing: XGBoost_Original

=== RESULT ===
           model architecture  parameters  accuracy  fdia_recall  fault_recall  model_size_kb  inference_time_ms
XGBoost_Original      XGBoost         NaN   0.99784     0.998915       0.99687     795.782227           0.000956

Testing: XGBoost_Pruned

=== RESULT ===
         model architecture  parameters  accuracy  fdia_recall  fault_recall  model_size_kb  inference_time_ms
XGBoost_Pruned      XGBoost         NaN  0.989815     0.997613      0.982786     121.142578            0.00048


# CNN

In [13]:
# CNN TESTS

X_test_cnn = np.transpose(X_test, (0, 2, 1))
X_cnn_timing = X_test_cnn[timing_idx]

cnn_models = {
    "CNN1D_Original": "CNN1D_HPO.keras",
    "CNN1D_Node": "CNN1D_NodePruned.keras",
    "CNN1D_Weight": "CNN1D_WeightPruned.keras",
    "CNN1D_WeightNode": "CNN1D_WeightNodePruned.keras",
    "CNN1D_NodeWeight": "CNN1D_NodeWeightPruned.keras"
}

for name, filename in cnn_models.items():

    print(f"\nTesting: {name}")

    model_path = MODELS_DIR / filename
    model = tf.keras.models.load_model(model_path, compile=False)

    # Full test set evaluation
    y_prob = model.predict(X_test_cnn, batch_size=len(X_test_cnn), verbose=0)
    y_pred = (y_prob > 0.5).astype(int).flatten()

    accuracy = accuracy_score(y_test, y_pred)
    fdia_recall = recall_score(y_test, y_pred, pos_label=1)
    fault_recall = recall_score(y_test, y_pred, pos_label=0)

    # Model information
    parameters = model.count_params()
    model_size_kb = get_model_size_kb(model)

    # Inference timing
    inference_time_ms = measure_inference_time(model, X_cnn_timing)

    result = {
        "model": name,
        "architecture": "CNN1D",
        "parameters": parameters,
        "accuracy": accuracy,
        "fdia_recall": fdia_recall,
        "fault_recall": fault_recall,
        "model_size_kb": model_size_kb,
        "inference_time_ms": inference_time_ms
    }

    results.append(result)

    print("\n=== RESULT ===")
    print(pd.DataFrame([result]).to_string(index=False))


Testing: CNN1D_Original
Runs: 100
Samples per run: 2430
Average inference time per sample: 0.071618 ms

=== RESULT ===
         model architecture  parameters  accuracy  fdia_recall  fault_recall  model_size_kb  inference_time_ms
CNN1D_Original        CNN1D       92289  0.998354     0.999349      0.997457     360.503906           0.071618

Testing: CNN1D_Node
Runs: 100
Samples per run: 2430
Average inference time per sample: 0.062879 ms

=== RESULT ===
     model architecture  parameters  accuracy  fdia_recall  fault_recall  model_size_kb  inference_time_ms
CNN1D_Node        CNN1D       46596  0.991667     0.999566      0.984546     182.015625           0.062879

Testing: CNN1D_Weight


KeyboardInterrupt: 

In [12]:
final_df = pd.DataFrame(results)

final_df = final_df[
    [
        "architecture",
        "model",
        "parameters",
        "accuracy",
        "fdia_recall",
        "fault_recall",
        "model_size_kb",
        "inference_time_ms"
    ]
]

final_df["accuracy"] = final_df["accuracy"].round(6)
final_df["fdia_recall"] = final_df["fdia_recall"].round(6)
final_df["fault_recall"] = final_df["fault_recall"].round(6)

# More decimal places
final_df["model_size_kb"] = final_df["model_size_kb"].round(6)
final_df["inference_time_ms"] = final_df["inference_time_ms"].round(6)

print("=== FINAL RESULTS ===")
display(final_df)

=== FINAL RESULTS ===


,architecture,model,parameters,accuracy,fdia_recall,fault_recall,model_size_kb,inference_time_ms
0,CNN1D,CNN1D_Original,92289,0.998354,0.999349,0.997457,360.503906,0.073971
1,CNN1D,CNN1D_Node,46596,0.991667,0.999566,0.984546,182.015625,0.059730
2,CNN1D,CNN1D_Weight,92289,0.999074,0.999349,0.998826,360.503906,0.070168
3,CNN1D,CNN1D_WeightNode,48751,0.992181,0.999566,0.985524,190.433594,0.055513
4,CNN1D,CNN1D_NodeWeight,46596,0.998765,0.999349,0.998239,182.015625,0.058061


In [25]:
mask = final_df["parameters"].notna()

final_df.loc[mask, "model_size_kb"] = (
    final_df.loc[mask, "parameters"] * 4 / 1024
)

final_df["model_size_kb"] = final_df["model_size_kb"].round(6)

display(final_df)

,architecture,model,parameters,accuracy,fdia_recall,fault_recall,model_size_kb,inference_time_ms
0,LSTM,LSTM_Original,159581.0,0.989198,0.987847,0.990415,623.363281,0.436676
1,LSTM,LSTM_Node,133697.0,0.983436,0.996311,0.971831,522.253906,0.414895
2,LSTM,LSTM_Weight,159581.0,0.994753,0.994792,0.994718,623.363281,0.434284
3,LSTM,LSTM_WeightNode,149537.0,0.994547,0.995877,0.993349,584.128906,0.421255
4,LSTM,LSTM_NodeWeight,133697.0,0.989300,0.993707,0.985329,522.253906,0.403727
5,MLP,MLP_Original,44753.0,0.998354,0.998264,0.998435,174.816406,0.034930
6,MLP,MLP_Node,28920.0,0.995988,0.994141,0.997653,112.968750,0.033342
7,MLP,MLP_Weight,44753.0,0.998148,0.998264,0.998044,174.816406,0.033445
8,MLP,MLP_WeightNode,28206.0,0.986214,0.971354,0.999609,110.179688,0.034538
9,MLP,MLP_NodeWeight,28920.0,0.997737,0.997613,0.997848,112.968750,0.033700


In [16]:
import subprocess
import sys
import numpy as np
import tempfile
from pathlib import Path

model_path = MODELS_DIR / "CNN1D_HPO.keras"
X_threads = X_cnn_timing

temp_path = Path(tempfile.gettempdir()) / "keras_thread_test.npy"
np.save(temp_path, X_threads)

script = """
import sys
import timeit
import numpy as np
import tensorflow as tf

threads = int(sys.argv[1])
model_path = sys.argv[2]
data_path = sys.argv[3]

tf.config.threading.set_intra_op_parallelism_threads(threads)
tf.config.threading.set_inter_op_parallelism_threads(threads)

X = np.load(data_path)
model = tf.keras.models.load_model(model_path, compile=False)

# Warm-up
model.predict(X, batch_size=len(X), verbose=0)

times = []

for _ in range(20):
    start = timeit.default_timer()

    model.predict(X, batch_size=len(X), verbose=0)

    end = timeit.default_timer()
    times.append(((end - start) * 1000) / len(X))

print(f"Threads {threads}: {np.mean(times):.6f} ms/sample")
"""

for threads in [1, 2, 4, 8]:
    result = subprocess.run(
        [sys.executable, "-c", script, str(threads), str(model_path), str(temp_path)],
        capture_output=True,
        text=True
    )

    print(result.stdout.strip())

Threads 1: 0.158439 ms/sample
Threads 2: 0.104548 ms/sample
Threads 4: 0.080819 ms/sample
Threads 8: 0.071471 ms/sample
